In [10]:
import sys
from pathlib import Path
import argparse
import contextlib
import json
import logging
import os
import sys
import eumdac
from datetime import datetime, timedelta, timezone
import shutil
import requests
import time
import pandas as pd 
import xarray as xr
import zipfile
from glob import glob
import re 

from dotenv import load_dotenv
load_dotenv(override=True)

PROJECT_PATH = Path(os.getenv("PROJECT_PATH"))
MTG_DATA_DIR      = PROJECT_PATH / "data" / "mtg"
MTG_CONSUMER_KEY=os.getenv("MTG_CONSUMER_KEY")
MTG_CONSUMER_SECRET=os.getenv("MTG_CONSUMER_SECRET")

lon_min = 10.0
lon_max = 30.0
lat_min = 40.0
lat_max = 60.0

In [2]:
credentials = (MTG_CONSUMER_KEY, MTG_CONSUMER_SECRET)
token = eumdac.AccessToken(credentials)

# Create datastore object with with your token
datastore = eumdac.DataStore(token)

In [ ]:
# Select an FCI collection, eg "FCI Level 1c High Resolution Image Data - MTG - 0 degree" - "EO:EUM:DAT:0782"
selected_collection = datastore.get_collection('EO:EUM:DAT:0782')

# Set sensing start and end time
start = datetime.now(timezone.utc) - timedelta(hours=3)
end = datetime.now(timezone.utc)

# Retrieve datasets that match the filter
products = selected_collection.search(
    dtstart=start,
    dtend=end)

products_processed = []

for product in products:

    with product.open() as source_file, open(os.path.join(MTG_DATA_DIR, source_file.name), mode="wb") as destination_file:
        shutil.copyfileobj(source_file, destination_file)


    products_processed.append({
        "name": source_file.name,
        'path': os.path.join(MTG_DATA_DIR, source_file.name),
        "sensing_start": product.sensing_start.isoformat(),
        "sensing_end": product.sensing_end.isoformat(),
    })
    print(f"Download of product {source_file.name} finished: {os.path.join(MTG_DATA_DIR, source_file.name)}")

df_products = pd.DataFrame(products_processed)

Download of product W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+LI-2-LGR--FD--x-x--ARC-x_C_EUMT_20260509102014_L2PF_OPE_20260509101000_20260509102000_N__O_0062_0000.zip finished: c:\projects\mapymeteo\data\mtg\W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+LI-2-LGR--FD--x-x--ARC-x_C_EUMT_20260509102014_L2PF_OPE_20260509101000_20260509102000_N__O_0062_0000.zip
Download of product W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+LI-2-LGR--FD--x-x--ARC-x_C_EUMT_20260509101014_L2PF_OPE_20260509100000_20260509101000_N__O_0061_0000.zip finished: c:\projects\mapymeteo\data\mtg\W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+LI-2-LGR--FD--x-x--ARC-x_C_EUMT_20260509101014_L2PF_OPE_20260509100000_20260509101000_N__O_0061_0000.zip
Download of product W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+LI-2-LGR--FD--x-x--ARC-x_C_EUMT_20260509100014_L2PF_OPE_20260509095000_20260509100000_N__O_0060_0000.zip finished: c:\projects\mapymeteo\data\mtg\W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+LI-2-LGR--FD--x-x--ARC-x_C_EUMT_20260509100014_L2PF_OPE_20260509095000

In [ ]:
#list of .nc files to process in folder MTG_DATA_DIR
nc_files = glob(os.path.join(MTG_DATA_DIR, "*0001.nc"))

df_lightnings = []

for nc_file in nc_files:
    ds = xr.open_dataset(nc_file)
    lat = ds['latitude'].values
    lon = ds['longitude'].values
    qa  = ds['group_filter_qa'].values
    group_time = ds['group_time'].values
    number_of_events = ds['number_of_events'].values
    radiance = ds['radiance'].values
    group_id = ds['group_id'].values

    df_group_frames = pd.DataFrame({
        'latitude': lat,
        'longitude': lon,
        'qa': qa,
        'group_time': group_time,
        'number_of_events': number_of_events,
        'radiance': radiance,
        'group_id': group_id
    })
    df_lightnings.append(df_group_frames)

df_lightnings = pd.concat(df_lightnings, ignore_index=True)
df_lightnings_extent = df_lightnings[(df_lightnings['latitude'] >= lat_min) & (df_lightnings['latitude'] <= lat_max) & (df_lightnings['longitude'] >= lon_min) & (df_lightnings['longitude'] <= lon_max)]

In [ ]:
#clean folder MTG_DATA_DIR - to keep last 3 hours data only